[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KoteshB/fde-project2-notebook-agentic-router/blob/main/Agentic_Router.ipynb)

# Deep Dive Agentic Retrieval Augmented Generation

An Agentic RAG is required when we use reasoning to determine which action(s) to take and in which order to take them. Essentially we use agents instead of a LLM directly to accomplish a set of tasks which requires planning, multi step reasoning, tool use and/or learning over time. Agents give us agency!

Agency : The ability to take action or to choose what action to take

In the context of RAG, we can plug in agents to enhance the reasoning prior to selection of RAG pipelines, within a RAG pipeline for retrieval or reranking and finally for synthesising before we send out the response. This improves RAG to a large extent by automating complex workflows and decisions that are required for a non trivial RAG use case.

### Purpose of this Agentic RAG
This notebook presents a practical implementation of Agentic Retrieval-Augmented Generation (RAG)—a system where decision-making and tool selection are delegated to an intelligent agent before executing a response. Rather than passing every query through a static RAG pipeline, this system introduces agency—the ability to choose the best course of action depending on the nature of the query.

At the heart of this implementation is a router prompt, which classifies user queries into one of three categories:

- OpenAI documentation: Queries related to tools, APIs, or usage guidelines for OpenAI models
- 10-K financial reports: Questions requiring retrieval from company filings or financial datasets
- Live Internet search: Broader, current, or comparative queries that need web access

Once the query is classified, the system invokes a corresponding route handler:

- For OpenAI and 10-K queries, it retrieves relevant context from a vector database (Qdrant) using text embeddings, then applies a RAG-based response generator.
- For Internet queries, it fetches real-time information using a web-access API (ARES).

This approach is an example of Agentic RAG, where reasoning precedes retrieval and generation. By plugging in agents before and within the RAG pipeline, we make the system smarter and more adaptive. This allows us to:

- Automatically choose the right retrieval method based on context
- Combine structured knowledge with real-time search
- Scale RAG beyond trivial use cases by integrating multi-step decision logic

Importantly, no external agentic frameworks are used—this is a ground-up implementation that demonstrates how to build a lightweight but intelligent agentic system using only a language model, prompt engineering, and retrieval tools.

## Setup and Dependencies

In [1]:
# Different versions of libraries for Colab vs local mac
try:
    from google.colab import userdata          
    IN_COLAB_IMPORTS = True
except ImportError:
    IN_COLAB_IMPORTS = False

if IN_COLAB_IMPORTS:
    # Install the necessary libraries
    !pip install openai
    !pip install qdrant_client
    !pip install transformers==4.48.0
else:
    print("Not in Colab")
    %pip install -q \
        openai \
        qdrant-client \
        "transformers==4.48.0" \
        "huggingface-hub==0.36.2"

Not in Colab

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install python-dotenv


if IN_COLAB_IMPORTS:
    %pip install matplotlib
    %pip install nest-asyncio
else:
    %pip install "numpy<2" "torch==2.2.2" "contourpy<1.4" transformers einops matplotlib python-dotenv nest-asyncio


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import basic libraries
import requests             # Used for making HTTP requests (e.g., calling ARES API for live internet queries)
import json                 # For parsing and structuring JSON data (especially OpenAI and routing responses)

# Credentials — Colab Secrets when on Colab, a local .env otherwise
try:
    from google.colab import userdata          # Colab: keys live in the 🔑 Secrets panel
    IN_COLAB = True
except ImportError:                            # Local Jupyter: keys live in .env
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
    IN_COLAB = False

    class userdata:                            # same .get() call works in both places
        @staticmethod
        def get(name):
            import os
            return os.getenv(name) or os.getenv(name.lower()) or os.getenv(
                name.replace("SERP_API_KEY", "SERPAPI_KEY"))

# OS operations
import os                   # Useful for accessing environment variables and managing paths

# OpenAI API client
from openai import OpenAI   # Official OpenAI client library to interface with GPT models for routing and generation

# Text processing
import re                   # Regular expressions for cleaning or preprocessing inputs (if needed)

# Optional visualization (for analysis/debugging purposes)
import matplotlib.pyplot as plt       # For displaying charts or visual debug outputs (e.g., embeddings visualizations)
import matplotlib.image as mpimg      # For loading/displaying images if needed (rare in RAG, but helpful in demos)

# Embedding models (used for text vectorization during retrieval)
from transformers import AutoTokenizer, AutoModel  # For loading custom transformer models if not using OpenAI embeddings

from qdrant_client import models

import qdrant_client
import asyncio
import nest_asyncio # Import nest_asyncio
nest_asyncio.apply() # Apply nest_asyncio to allow nested event loops
# # Vector database client
# from qdrant_client import QdrantClient   # Qdrant is used as the vector store to retrieve documents based on similarity

/Users/kotesh/projects/fde-hamza/fde-project2-notebook-agentic-router/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Defining the Internet Tool

First, we will define a tool function that enables our system to answer queries requiring real-time, internet-based information. Not all questions can be answered using static documents like OpenAI docs or financial filings—sometimes users ask about current trends, comparisons, or live updates.

To handle this, we introduce a live search capability using the **SerpApi**.

### What is SerpApi?  
SerpApi is a Google Search API that allows you to:

- Search the internet in real time using Google.
- Get structured results including answer boxes, organic results, and snippets.

This is particularly useful for questions about:

- Current events (e.g., *"Latest AI tools in 2025"*),
- Tech comparisons (e.g., *"Gemini vs GPT-4"*),
- General knowledge outside internal datasets.

Please generate the API key [here](https://serpapi.com)


In [4]:
#loads serp api key from colab secrets
serp_api_key=userdata.get('SERP_API_KEY')

In [5]:
import requests  # For sending HTTP requests to the SerpApi

def get_internet_content(user_query: str, action: str):
    """
    Fetches a response from the internet using SerpApi based on the user's query.

    This function serves as the tool invoked when the router classifies a query
    as requiring real-time information beyond internal datasets—i.e., "INTERNET_QUERY".
    It sends the query to SerpApi (Google Search) and returns structured results.

    Args:
        user_query (str): The user's question that needs a live answer.
        action (str): Route type (always expected to be "INTERNET_QUERY").

    Returns:
        str: Response text from live Google search results or an error message.
    """
    print("Getting your response from the internet 🌐 ...")

    params = {
        "q": user_query,
        "api_key": serp_api_key,
        "engine": "google",
        "num": 5,
    }

    try:
        print(f"calling INTERNET search using SERPAPI for: {user_query}")
        response = requests.get("https://serpapi.com/search.json", params=params)
        response.raise_for_status()
        data = response.json()

        parts = []

        # Answer box — Google's highlighted direct answer (most relevant)
        answer_box = data.get("answer_box", {})
        if answer_box.get("answer"):
            parts.append(f"[Direct Answer] {answer_box['answer']}")
        elif answer_box.get("snippet"):
            parts.append(f"[Direct Answer] {answer_box['snippet']}")

        # Top organic results — titles + snippets
        for i, result in enumerate(data.get("organic_results", [])[:5], start=1):
            title = result.get("title", "")
            snippet = result.get("snippet", "")
            link = result.get("link", "")
            if snippet:
                parts.append(f"[{i}] {title}\n    {snippet}\n    Source: {link}")

        if not parts:
            return "No results found."

        return "\n\n".join(parts)

    # Handle HTTP-level errors (e.g., 400s or 500s)
    except requests.exceptions.HTTPError as http_err:
        return f"HTTP error occurred: {http_err}"

    # Handle general connection, timeout, or request formatting issues
    except requests.exceptions.RequestException as req_err:
        return f"Request error occurred: {req_err}"

    # Catch-all for any unexpected failure
    except Exception as err:
        return f"An unexpected error occurred: {err}"


In [6]:
print(get_internet_content("Tell me about best travel destinations in 2026?","INTERNET_QUERY")) #run internet function to test results

Getting your response from the internet 🌐 ...
calling INTERNET search using SERPAPI for: Tell me about best travel destinations in 2026?
[1] Best Places to Travel in 2026: 10 Emerging Destinations to ...
    This is my travel guide for 2026 including the best places to travel in 2026. From Albania and Sri Lanka to Taiwan and Kyrgyzstan!
    Source: https://girlonazebra.com/best-places-to-travel-in-2026/

[2] 11 Unique Travel Destinations To Visit In 2026
    Unique Travel Destinations to Visit in 2026 · Jacksonville, Oregon · Aarhus, Denmark · Spokane, Washington · Oceanside, California · Jasper, Canada.
    Source: https://www.forbes.com/sites/noelburgess/2025/12/28/11-unique-travel-destinations-to-visit-in-2026/

[3] The 6 Best Places To Travel In 2026, According To A ...
    The 6 Best Places To Travel In 2026, According To A Travel Advisor · Nikko, Japan · Kona, Hawaii · Versailles, France · Guanacaste, Costa Rica.
    Source: https://www.forbes.com/sites/christopherelliott/2025/10

## 2. Router Query Function — Giving the Agent Its Brain

In this step, we will define the router function, which plays a critical role in our Agentic RAG system.

### What is a Router?

A router is like the decision-making brain of our assistant.

Before trying to answer a user's question, the system first needs to figure out:

> “Where should I go to find the right answer?”

To make this decision, we use the OpenAI GPT model. We provide it with a detailed system prompt that explains how to classify the user's question into one of these categories:

- **OPENAI_QUERY** → Questions about OpenAI tools, APIs, models, or documentation.
- **10K_DOCUMENT_QUERY** → Questions about companies, financial filings, or analysis based on 10-K reports.
- **INTERNET_QUERY** → Anything else that likely requires real-time or general web information.

### What does the function do?

- Sends the user's question to the OpenAI API.
- Receives a JSON response containing:
  - `action`: The category the query belongs to.
  - `reason`: A short explanation for the decision.
  - `answer`: (Optional) A quick response if it’s simple enough (left blank for internet queries).
- Parses the response and returns it as a Python dictionary.

### Why is this important?

This router gives the system agency—the ability to decide which knowledge source to use. It’s what makes this pipeline agentic, not just static.

Without the router, every query would follow the same path. With it, we can:

- Dynamically switch between tools and data sources.
- Handle different types of user questions intelligently.
- Avoid wasting resources on unnecessary steps.


## Query Routing Workflow

The diagram below shows the full decision flow — from receiving a user query to returning a final response.

```
                        ┌─────────────────────┐
                        │     User Query      │
                        └──────────┬──────────┘
                                   │
                                   ▼
                    ┌──────────────────────────────┐
                    │      Router LLM (GPT-4o)     │
                    │         route_query()         │
                    │                              │
                    │  Reads the query and decides │
                    │  which data source to use    │
                    └──────────────┬───────────────┘
                                   │
           ┌───────────────────────┼───────────────────────┐
           │                       │                       │
           ▼                       ▼                       ▼
┌─────────────────────┐ ┌─────────────────────┐ ┌─────────────────────┐
│    OPENAI_QUERY     │ │ 10K_DOCUMENT_QUERY  │ │   INTERNET_QUERY    │
│                     │ │                     │ │                     │
│ e.g. "What are      │ │ e.g. "What was      │ │ e.g. "Best LLMs     │
│  OpenAI Agents?"    │ │  Uber's revenue?"   │ │  in 2026?"          │
└──────────┬──────────┘ └──────────┬──────────┘ └──────────┬──────────┘
           │                       │                        │
           ▼                       ▼                        ▼
┌─────────────────────┐ ┌─────────────────────┐  ┌──────────────────────┐
│   Embed Query       │ │   Embed Query        │  │      SerpApi         │
│  (Nomic Model)      │ │  (Nomic Model)       │  │  get_internet_       │
│                     │ │                      │  │  content()           │
│  get_text_          │ │  get_text_           │  │                      │
│  embeddings()       │ │  embeddings()        │  │  Live Google search  │
└──────────┬──────────┘ └──────────┬──────────┘  └──────────┬───────────┘
           │                       │                          │
           ▼                       ▼                          │
┌─────────────────────┐ ┌─────────────────────┐              │
│  Qdrant Vector DB   │ │  Qdrant Vector DB   │              │
│  Collection:        │ │  Collection:         │              │
│  "opnai_data"       │ │  "10k_data"          │              │
│                     │ │                      │              │
│  Retrieve top-3     │ │  Retrieve top-3      │              │
│  similar chunks     │ │  similar chunks      │              │
└──────────┬──────────┘ └──────────┬──────────┘              │
           │                       │                          │
           └───────────┬───────────┘                          │
                       ▼                                      │
           ┌───────────────────────┐                          │
           │    RAG Response       │                          │
           │    Generator          │                          │
           │  rag_formatted_       │                          │
           │  response()           │                          │
           │                       │                          │
           │  GPT-4 synthesizes    │                          │
           │  answer from context  │                          │
           │  + adds citations     │                          │
           └───────────┬───────────┘                          │
                       │                                      │
                       └──────────────────┬───────────────────┘
                                          ▼
                             ┌────────────────────────┐
                             │     Final Response     │
                             │       to User          │
                             └────────────────────────┘
```

### Key decision points at a glance

| Route | Trigger | Retrieval Method | Response Generator |
|---|---|---|---|
| `OPENAI_QUERY` | OpenAI docs, APIs, Agents | Qdrant `opnai_data` (top-3 chunks) | `rag_formatted_response()` via GPT-4 |
| `10K_DOCUMENT_QUERY` | Financial filings, company revenue | Qdrant `10k_data` (top-3 chunks) | `rag_formatted_response()` via GPT-4 |
| `INTERNET_QUERY` | Anything else / real-time info | SerpApi live Google search | Raw search result snippets |

> **Note:** Both vector-based routes share the same embedding model (`nomic-embed-text-v1.5`) and RAG generator — only the Qdrant collection changes. The router's JSON output (`action` field) is the single decision variable that drives the entire flow.


In [7]:
# Securely retrieve the OpenAI API key from Colab's user data store
# This avoids hardcoding sensitive credentials directly in the notebook
openai_api_key = userdata.get('OPENAI_API_KEY')

# Initialize the OpenAI client with the retrieved API key
# This client will be used for:
# - Query classification via the router prompt
# - Potentially generating responses from retrieved context
openaiclient = OpenAI(api_key=openai_api_key)


In [8]:
from openai import OpenAIError

def route_query(user_query: str):
    router_system_prompt =f"""
    As a professional query router, your objective is to correctly classify user input into one of three categories based on the source most relevant for answering the query:
    1. "OPENAI_QUERY": If the user's query appears to be answerable using information from OpenAI's official documentation about Agents, tools, models, APIs, or services (e.g., guardrails, agents, what is an agent, embeddings, moderation API, usage guidelines).
    2. "10K_DOCUMENT_QUERY": If the user's query pertains to a collection of documents from the 10k annual reports, datasets, or other structured documents, typically for research, analysis, or financial content.
    3. "INTERNET_QUERY": If the query is neither related to OpenAI nor the 10k documents specifically, or if the information might require a broader search (e.g., news, trends, tools outside these platforms), route it here.

    Your decision should be made by assessing the domain of the query.

    Always respond in this valid JSON format:
    {{
        "action": "OPENAI_QUERY" or "10K_DOCUMENT_QUERY" or "INTERNET_QUERY",
        "reason": "brief justification",
        "answer": "AT MAX 5 words answer. Leave empty if INTERNET_QUERY"
    }}

    EXAMPLES:

    - User: "How to fine-tune GPT-3?"
    Response:
    {{
        "action": "OPENAI_QUERY",
        "reason": "Fine-tuning is OpenAI-specific",
        "answer": "Use fine-tuning API"
    }}

    - User: "Where can I find the latest financial reports for the last 10 years?"
    Response:
    {{
        "action": "10K_DOCUMENT_QUERY",
        "reason": "Query related to annual reports",
        "answer": "Access through document database"
    }}

    - User: "Top leadership styles in 2024"
    Response:
    {{
        "action": "INTERNET_QUERY",
        "reason": "Needs current leadership trends",
        "answer": ""
    }}

    - User: "What's the difference between ChatGPT and Claude?"
    Response:
    {{
        "action": "INTERNET_QUERY",
        "reason": "Cross-comparison of different providers",
        "answer": ""
    }}

    Strictly follow this format for every query, and never deviate.
    User: {user_query}
    """

    try:
        print(f"route_query: OPENAI (LLM) call to find the appropriate route for query: {user_query}")
        # Query the GPT-4 model with the router prompt and user input
        response = openaiclient.chat.completions.create(
            model="gpt-5.6-luna",
            messages=[{"role": "system", "content": router_system_prompt}]
        )

        # Extract and parse the model's JSON response
        task_response = response.choices[0].message.content
        json_match = re.search(r"\{.*\}", task_response, re.DOTALL)
        json_text = json_match.group()
        parsed_response = json.loads(json_text)
        print(f"route_query response: {parsed_response}")
        return parsed_response

    # Handle OpenAI API errors (e.g., rate limits, authentication)
    except OpenAIError as api_err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"OpenAI API error: {api_err}",
            "answer": ""
        }

    # Handle case where model response isn't valid JSON
    except json.JSONDecodeError as json_err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"JSON parsing error: {json_err}",
            "answer": ""
        }

    # Catch-all for any other unforeseen issues
    except Exception as err:
        return {
            "action": "INTERNET_QUERY",
            "reason": f"Unexpected error: {err}",
            "answer": ""
        }

In [9]:
route_query("what is the revenue of uber in 2021?")


route_query: OPENAI (LLM) call to find the appropriate route for query: what is the revenue of uber in 2021?
route_query response: {'action': '10K_DOCUMENT_QUERY', 'reason': 'Queries company revenue from an annual financial report', 'answer': "Uber's 2021 revenue"}


{'action': '10K_DOCUMENT_QUERY',
 'reason': 'Queries company revenue from an annual financial report',
 'answer': "Uber's 2021 revenue"}

In [10]:
route_query("what is an AI Agent?")

route_query: OPENAI (LLM) call to find the appropriate route for query: what is an AI Agent?
route_query response: {'action': 'OPENAI_QUERY', 'reason': 'AI agents are covered in OpenAI documentation', 'answer': 'Software that performs tasks'}


{'action': 'OPENAI_QUERY',
 'reason': 'AI agents are covered in OpenAI documentation',
 'answer': 'Software that performs tasks'}

## 3. Setting Up Qdrant Vector Database for Agentic RAG
In this step, we are connecting our agent to a pre-built vector database using Qdrant—a tool used to store and search document embeddings (numerical representations of text).

What Are We Doing?
We are loading an existing Qdrant database that was downloaded from a GitHub repository. This database already contains:

- Vectorized OpenAI documentation
- Vectorized 10-K financial filings

By loading this saved data:

- We save time (no need to re-embed the documents)
- We enable fast similarity search to retrieve relevant text chunks

This setup allows our system to perform semantic search, meaning it can understand the meaning of the user query and match it with the most relevant pieces of information stored in the database.


### Why This Matters in Agentic RAG
Once the router decides that the query should go to the OpenAI docs or the 10-K reports, our system uses Qdrant to:

- Search for the most relevant pieces of text
- Pass those to the model to generate a grounded answer

So, this step is essential to support retrieval-augmented generation (RAG) within our agentic flow.

#Data Sources:

**10K Database: Lyft 2024 & Uber 2021 SEC filings**

**OpenAI Docs: Official OpenAI documentation about Agents**

For lecture demo purposes, the vecitr database has already been created and hosted on Github which we will clone here. In order to create your own embeddings, the notebook and data will be hosted and shared on github

In [11]:
# Colab only — wipe a previous clone if you need a clean copy
if IN_COLAB:
    !rm -rf /content/multi-agent-course

In [12]:
import os

# The prebuilt Qdrant collections (10-K + OpenAI docs) ship with the repo.
# On Colab we clone to get them; locally you already have them.
if IN_COLAB:
    print("running in Google Colab")
    !git clone https://github.com/hamzafarooq/multi-agent-course.git
else:
    print("NOT running in Google Colab")
    if os.path.isdir("multi-agent-course"):
        print("multi-agent-course directory exists. do not need to clone")
    else:
        print("multi-agent-course does not exist. clone from github")
        !git clone https://github.com/hamzafarooq/multi-agent-course.git

NOT running in Google Colab
multi-agent-course directory exists. do not need to clone


In [13]:
# 🗄️ Initializing Qdrant client with the local path to the vector database
# Prebuilt collections (10-K and OpenAI docs) — cloned on Colab, already present locally.
import os

_MODULE = "modules/Module_3_Production_Agentic_RAG_AI_Systems"
if IN_COLAB:
    QDRANT_PATH = f"/content/multi-agent-course/{_MODULE}/Agentic_RAG/qdrant_data"
else:
    # the notebook lives in the module folder, so the data sits right next to it
    # QDRANT_PATH = os.path.join(os.getcwd(), "Agentic_RAG", "qdrant_data")
    QDRANT_PATH = os.path.join(os.getcwd(), "multi-agent-course", _MODULE, "Agentic_RAG", "qdrant_data")

print("Qdrant path:", QDRANT_PATH)
client = qdrant_client.AsyncQdrantClient(path=QDRANT_PATH)

Qdrant path: /Users/kotesh/projects/fde-hamza/fde-project2-notebook-agentic-router/multi-agent-course/modules/Module_3_Production_Agentic_RAG_AI_Systems/Agentic_RAG/qdrant_data


## 4. Building the Retriever and RAG for Vector Databases
In this section, we build the core logic that allows our agent to find relevant documents and generate grounded answers using them.

###Step 1: Import the Embedding Model
We start by importing the nomic-ai/nomic-embed-text-v1.5 model from Hugging Face. This model is used to convert any text (such as a user query) into a dense vector, known as an embedding. These embeddings capture the semantic meaning of text, allowing us to later compare and retrieve similar documents.


In [14]:
import platform, sys
print(platform.machine(), sys.version)
print(sys.executable)

# due to intel chip on mac, need specific versions 
# %pip install "torch==2.2.2" "numpy<2"
# %pip install einops

x86_64 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 08:28:27) [Clang 14.0.6 ]
/Users/kotesh/projects/fde-hamza/fde-project2-notebook-agentic-router/.venv/bin/python


In [15]:
if not IN_COLAB_IMPORTS:
    %pip install "numpy<2"


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
# Load the tokenizer and embedding model from Hugging Face
# This model converts raw text into dense vector representations (embeddings)
# Used for similarity search in Qdrant during document retrieval
text_tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

def get_text_embeddings(text):
    """
    Converts input text into a dense embedding using the Nomic embedding model.
    These embeddings are used to query Qdrant for semantically relevant document chunks.

    Args:
        text (str): The input text or query from the user.

    Returns:
        np.ndarray: A fixed-size vector representing the semantic meaning of the input.
    """
    # Tokenize and prepare input for the model
    inputs = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    # Forward pass to get model outputs
    outputs = text_model(**inputs)

    # Take the mean across all token embeddings to get a single vector (pooled representation)
    embeddings = outputs.last_hidden_state.mean(dim=1)

    # Convert to NumPy array and detach from computation graph
    return embeddings[0].detach().numpy()

# Example usage: Generate and preview the embedding of a test sentence
text = "This is a test sentence."
embeddings = get_text_embeddings(text)
print(embeddings[:5])  # Print first 5 dimensions for inspection


<All keys matched successfully>


[ 1.279969    0.4015836  -3.5162654  -0.39813295  1.5919135 ]


### Step 2: Define the Embedding Function
We then define a function get_text_embeddings() which:

- Tokenizes the input text
- Runs it through the model
- Computes the average of all token embeddings
- Returns a single vector that represents the full sentence

This vector will be used to query Qdrant to find the most relevant document chunks based on similarity.

In [17]:
def rag_formatted_response(user_query: str, context: list):
    """
    Generate a response to the user query using the provided context,
    with article references formatted as [1][2], etc.

    This function performs the final step in the RAG pipeline—synthesizing an answer
    from retrieved document chunks (context). It prompts the model to generate a
    grounded response, explicitly citing sources using a reference format.

    Args:
        user_query (str): The user's original question.
        context (list): List of text chunks retrieved from Qdrant (10-K or OpenAI docs).

    Returns:
        str: A generated response grounded in the retrieved context, with numbered citations.
    """

    # Construct a RAG prompt that includes both:
    # 1. The user's query
    # 2. The supporting context documents
    # The prompt instructs the model to answer using only the provided context,
    # and to include citations like [1], [2], etc. based on chunk IDs or order.
    rag_prompt = f"""
       Based on the given context, answer the user query: {user_query}\nContext:\n{context}
       and employ references to the ID of articles provided [ID], ensuring their relevance to the query.
       The referencing should always be in the format of [1][2]... etc. </instructions>
    """

    #  Call GPT-5.6-Luna to generate the response using the RAG-style prompt
    print(f"rag_formatted_response: OPENAI (llm) call with user_query and context")
    response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": rag_prompt},
        ]
    )

    # Return the model's generated answer
    return response.choices[0].message.content


### Step 3: Define the RAG Response Generator
After retrieving relevant text chunks from Qdrant, we use the rag_formatted_response() function to generate a final answer. This function:

- Takes the user query and the retrieved document chunks
- Builds a prompt that asks the language model (GPT-5.6-Luna) to answer the question using only the provided context
- Instructs the model to include references like [1], [2] for traceability

This ensures the output is not only informative but also grounded in actual retrieved data.

Together, these two functions lay the foundation for combining retrieval (from vector DB) and generation (from LLM) — the two pillars of a RAG system.



In [18]:
async def retrieve_and_response(user_query: str, action: str):
    """
    Retrieves relevant text chunks from the appropriate Qdrant collection
    based on the query type, then generates a response using RAG.

    This function powers the retrieval and response generation pipeline
    for queries that are classified as either OPENAI-related or 10-K related.
    It uses semantic search to fetch relevant context from a Qdrant vector store
    and then generates a response using that context via a RAG prompt.

    Args:
        user_query (str): The user's input question.
        action (str): The classification label from the router (e.g., "OPENAI_QUERY", "10K_DOCUMENT_QUERY").

    Returns:
        str: A model-generated response grounded in retrieved documents, or an error message.
    """

    # Define mapping of routing labels to their respective Qdrant collections
    collections = {
        "OPENAI_QUERY": "opnai_data",           # Collection of OpenAI documentation embeddings
        "10K_DOCUMENT_QUERY": "10k_data"        # Collection of 10-K financial document embeddings
    }

    try:
        # Ensure that the provided action is valid
        if action not in collections:
            return "Invalid action type for retrieval."

        # Step 1: Convert the user query into a dense vector (embedding)
        try:
            print(f"retrieve_and_response: get_text_embeddings for user_query")
            query = get_text_embeddings(user_query)
        except Exception as embed_err:
            return f"Embedding error: {embed_err}"  # Fail early if embedding fails

        # Step 2: Retrieve top-matching chunks from the relevant Qdrant collection
        try:
            print(f"retrieve_and_response: get context (text hits and references) for the user_query embeddings from qdrant")
            text_hits = await client.query_points(
                collection_name=collections[action],  # Choose the right collection based on routing
                query=query,                          # The embedding of the user's query
                limit=3                               # Fetch top 3 relevant chunks
            )
        except Exception as qdrant_err:
            return f"Vector DB query error: {qdrant_err}"  # Handle Qdrant access issues

        # Extract the raw content from the retrieved vector hits
        contents = [point.payload['content'] for point in text_hits.points]

        # If no relevant content is found, return early
        if not contents:
            return "No relevant content found in the database."

        # Step 3: Pass the retrieved context to the RAG model to generate a response
        try:
            print("retrieve_and_response: call rag_formatted_response and pass user_query and context")
            response = rag_formatted_response(user_query, contents)
            return response
        except Exception as rag_err:
            return f"RAG response error: {rag_err}"  # Handle generation failures

    # Catch any unforeseen errors in the overall process
    except Exception as err:
        return f"Unexpected error: {err}"


# 5. Putting It All Together: Running the Agentic RAG
In this final step, we combine everything into a single function that controls the entire Agentic RAG workflow. The agentic_rag() function acts as the main orchestrator of the system.

Here’s what it does:

- Prints the user's query for reference.
- Uses the router function (powered by GPT) to decide which type of data source to use:
  - OpenAI documentation
  - 10-K financial reports
- Internet search
- Calls the correct function based on the route:
- If it’s an OpenAI or 10-K query, it retrieves data from Qdrant and generates a RAG response.
- If it’s an Internet query, it uses the ARES API to fetch live information.
- Displays the final response, neatly formatted in the console.

This step brings the agentic loop full circle—from understanding the question, reasoning about where to search, to finally responding with the best possible answer.

In [19]:
# Dictionary that maps the route labels (decided by the router) to their respective functions
# Each type of query is handled differently:
# - OPENAI_QUERY and 10K_DOCUMENT_QUERY use document retrieval + RAG
# - INTERNET_QUERY uses a web search API
routes = {
    "OPENAI_QUERY": retrieve_and_response,
    "10K_DOCUMENT_QUERY": retrieve_and_response,
    "INTERNET_QUERY": get_internet_content,
}

def agentic_rag(user_query: str):
    """
    Main function that runs the full Agentic RAG system.

    This function takes a user's question, decides what type of query it is (OpenAI-related,
    financial document-related, or general internet), and then calls the right function
    to handle it. Finally, it prints out the full conversation and response.

    Args:
        user_query (str): The user's input question.

    Returns:
        None (It just prints the result nicely to the console)
    """

    #  Terminal color codes to make the printed output easier to read and visually structured
    CYAN = "\033[96m"
    GREY = "\033[90m"
    BOLD = "\033[1m"
    RESET = "\033[0m"

    try:
        # Step 1: Print the user's original question to the console
        print(f"{BOLD}{CYAN}👤 User Query:{RESET} {user_query}\n")

        # Step 2: Use the router (powered by GPT) to decide which route the query belongs to
        try:
            response = route_query(user_query)
        except Exception as route_err:
            # If something goes wrong while classifying the query, show an error message
            print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
            print(f"Routing error: {route_err}\n")
            return

        # Extract the routing decision and the reason behind it
        action = response.get("action")  # e.g., "OPENAI_QUERY"
        reason = response.get("reason")  # e.g., "Related to OpenAI tools"

        # Step 3: Show the selected route and why it was chosen
        print(f"{GREY}📍 Selected Route: {action}")
        print(f"📝 Reason: {reason}")
        print(f"⚙️ Processing query...{RESET}\n")

        # Step 4: Call the correct function depending on the route (retrieval or web search)
        try:
            route_function = routes.get(action)  # Find the function to use for this route
            if route_function:
                if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
                    # Use asyncio.run for async functions, nest_asyncio will handle nested loops
                    result = asyncio.run(route_function(user_query, action))
                else:
                    # Otherwise, call it directly (e.g., get_internet_content is synchronous)
                    result = route_function(user_query, action)
            else:
                result = f"Unsupported action: {action}"  # Catch unknown routing types
        except Exception as exec_err:
            result = f"Execution error: {exec_err}"  # Handle failure in the chosen route function

        # Step 5: Print the final response to the user
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"{result}\n")

    except Exception as err:
        # Catch-all for any unexpected errors in the overall logic
        print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
        print(f"Unexpected error occurred: {err}\n")


In [20]:
agentic_rag("what was uber revenue in 2021?")

👤 User Query: what was uber revenue in 2021?

route_query: OPENAI (LLM) call to find the appropriate route for query: what was uber revenue in 2021?
route_query response: {'action': '10K_DOCUMENT_QUERY', 'reason': 'Revenue question about a company’s annual financial report', 'answer': "Uber's 2021 revenue was $17.5 billion"}
📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Revenue question about a company’s annual financial report
⚙️ Processing query...

retrieve_and_response: get_text_embeddings for user_query
retrieve_and_response: get context (text hits and references) for the user_query embeddings from qdrant
retrieve_and_response: call rag_formatted_response and pass user_query and context
rag_formatted_response: OPENAI (llm) call with user_query and context
🤖 BOT RESPONSE:

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]



In [21]:
agentic_rag("what was lyft revenue in 2022?")

👤 User Query: what was lyft revenue in 2022?

route_query: OPENAI (LLM) call to find the appropriate route for query: what was lyft revenue in 2022?
route_query response: {'action': '10K_DOCUMENT_QUERY', 'reason': "Lyft's 2022 revenue is reported in its annual financial filings", 'answer': 'Approximately $4.1 billion'}
📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Lyft's 2022 revenue is reported in its annual financial filings
⚙️ Processing query...

retrieve_and_response: get_text_embeddings for user_query
retrieve_and_response: get context (text hits and references) for the user_query embeddings from qdrant
retrieve_and_response: call rag_formatted_response and pass user_query and context
rag_formatted_response: OPENAI (llm) call with user_query and context
🤖 BOT RESPONSE:

Lyft’s revenue in 2022 was **$4.095 billion** (reported as **$4,095,135 thousand**) [1].



In [22]:
agentic_rag("List me down new LLMs in 2025")

👤 User Query: List me down new LLMs in 2025

route_query: OPENAI (LLM) call to find the appropriate route for query: List me down new LLMs in 2025
route_query response: {'action': 'INTERNET_QUERY', 'reason': 'Requests current information about newly released LLMs', 'answer': ''}
📍 Selected Route: INTERNET_QUERY
📝 Reason: Requests current information about newly released LLMs
⚙️ Processing query...

Getting your response from the internet 🌐 ...
calling INTERNET search using SERPAPI for: List me down new LLMs in 2025
🤖 BOT RESPONSE:

[1] LLM Tools Cheat Sheet 2025 | Manas Dasgupta
    LLM Tools Cheat Sheet 2025 A quick-reference guide for developers, builders, and AI enthusiasts. Streamline your AI stack with the best ...
    Source: https://www.linkedin.com/posts/manasdasgupta_llm-tools-cheat-sheet-2025-a-quick-reference-activity-7356524736165654529-c2dC

[2] LLM Deep Dive 2025: Why Claude 4 and GPT-5.1 Change ...
    The LLM landscape in late 2025 is a dynamic ecosystem, far removed fr

In [23]:
agentic_rag("how to work with chat completions?")

👤 User Query: how to work with chat completions?

route_query: OPENAI (LLM) call to find the appropriate route for query: how to work with chat completions?
route_query response: {'action': 'OPENAI_QUERY', 'reason': 'Chat Completions is an OpenAI API', 'answer': 'Use the Chat Completions API'}
📍 Selected Route: OPENAI_QUERY
📝 Reason: Chat Completions is an OpenAI API
⚙️ Processing query...

retrieve_and_response: get_text_embeddings for user_query
retrieve_and_response: get context (text hits and references) for the user_query embeddings from qdrant
retrieve_and_response: call rag_formatted_response and pass user_query and context
rag_formatted_response: OPENAI (llm) call with user_query and context
🤖 BOT RESPONSE:

Chat Completions let you send a sequence of messages to a model and receive an assistant response. The basic workflow is:

1. Create a client.
2. Provide messages with roles such as `system`, `user`, and `assistant`.
3. Read the returned assistant message.
4. Preserve the c

In [24]:
agentic_rag("best ways to build Agents")

👤 User Query: best ways to build Agents

route_query: OPENAI (LLM) call to find the appropriate route for query: best ways to build Agents
route_query response: {'action': 'OPENAI_QUERY', 'reason': 'Building agents relates to OpenAI agent documentation', 'answer': 'Use the Agents SDK'}
📍 Selected Route: OPENAI_QUERY
📝 Reason: Building agents relates to OpenAI agent documentation
⚙️ Processing query...

retrieve_and_response: get_text_embeddings for user_query
retrieve_and_response: get context (text hits and references) for the user_query embeddings from qdrant
retrieve_and_response: call rag_formatted_response and pass user_query and context
rag_formatted_response: OPENAI (llm) call with user_query and context
🤖 BOT RESPONSE:

## Best ways to build agents

1. **Use agents for genuinely multi-step workflows**  
   Build an agent when the system must independently decide what to do, use tools, adapt to failures, and determine when a task is complete. Simple chatbots, single-turn generat

## 6. Role-Based Access Control (RBAC)

Everything so far assumes one kind of user: whoever asks gets whatever the router
picks. In a real deployment that's rarely true. An engineer shouldn't be able to pull
finance's 10-K numbers out of the vector store, and a finance analyst has no business
reading internal engineering docs — even though both are talking to the same agent.

RBAC puts a **permission check between the router's decision and the tool call**. The
router still reasons about *where* the answer lives; RBAC decides whether *this user*
is allowed to go there. If not, the request is rejected before any embedding, vector
search, or grounding call happens.

**This demo — 2 roles, 3 knowledge sources:**

| Knowledge source | Route label | `engineer` | `finance_analyst` |
|---|---|---|---|
| 📘 OpenAI documentation (Qdrant) | `OPENAI_QUERY` | ✅ | ✅ |
| 📗 10-K filings (Qdrant) | `10K_DOCUMENT_QUERY` | ❌ | ✅ |
| 🌐 Live internet search (SerpApi) | `INTERNET_QUERY` | ✅ | ❌ |

The two Qdrant collections and the SerpApi tool are the same ones built above — RBAC
is a layer on top, not a different pipeline.


In [25]:
# ── Users → role ─────────────────────────────────────────────────────────────
# Stand-in for a real identity provider. In production this comes from SSO/JWT
# claims or an internal users table — never a dict in the notebook.
USERS = {
    "alice": "engineer",
    "bob":   "finance_analyst",
}

# ── Roles → the route labels each role may reach ─────────────────────────────
# This is an allow-list: anything not listed here is denied by default.
ROLE_PERMISSIONS = {
    "engineer":        {"OPENAI_QUERY", "INTERNET_QUERY"},
    "finance_analyst": {"OPENAI_QUERY", "10K_DOCUMENT_QUERY"},
}

# Human-readable names, used only for clearer denial messages
SOURCE_LABELS = {
    "OPENAI_QUERY":       "OpenAI documentation",
    "10K_DOCUMENT_QUERY": "10-K financial filings",
    "INTERNET_QUERY":     "live internet search",
}


def has_access(user_id: str, action: str) -> bool:
    """True only if this user's role is explicitly allowed to use this route."""
    role = USERS.get(user_id)
    return role is not None and action in ROLE_PERMISSIONS.get(role, set())


def allowed_sources(user_id: str) -> set:
    """Every route label this user may reach — useful for constraining the router."""
    return ROLE_PERMISSIONS.get(USERS.get(user_id), set())


print("alice  (engineer)        →", allowed_sources("alice"))
print("bob    (finance_analyst) →", allowed_sources("bob"))
print("carol  (unknown user)    →", allowed_sources("carol"))

alice  (engineer)        → {'INTERNET_QUERY', 'OPENAI_QUERY'}
bob    (finance_analyst) → {'10K_DOCUMENT_QUERY', 'OPENAI_QUERY'}
carol  (unknown user)    → set()


In [26]:
def secure_agentic_rag(user_id: str, user_query: str):
    """
    The same agentic RAG loop as above, with one addition: after the router picks a
    route, the user's role must permit that route before the tool is called.

    Order of operations:
        1. Identify the user  → unknown users are rejected immediately
        2. Route the query    → router decides which knowledge source fits
        3. RBAC check         → role allowed to use that source? deny if not
        4. Retrieve + answer  → only ever reached by an authorized request

    Args:
        user_id (str): Who is asking (looked up in USERS).
        user_query (str): The question.

    Returns:
        str: The answer, or a denial message.
    """
    CYAN, GREY, RED, GREEN, BOLD, RESET = (
        "\033[96m", "\033[90m", "\033[91m", "\033[92m", "\033[1m", "\033[0m"
    )

    role = USERS.get(user_id)
    print(f"{BOLD}{CYAN}👤 User:{RESET} {user_id}  (role: {role or 'UNKNOWN'})")
    print(f"{BOLD}{CYAN}❓ Query:{RESET} {user_query}\n")

    # Step 1 — unknown identity is denied before anything else runs
    if role is None:
        print(f"{RED}🚫 ACCESS DENIED{RESET} — unknown user '{user_id}'.\n")
        return f"🚫 Access denied: unknown user '{user_id}'."

    # Step 2 — the router still does the reasoning about where the answer lives
    try:
        decision = route_query(user_query)
    except Exception as route_err:
        return f"Routing error: {route_err}"

    action = decision.get("action")
    reason = decision.get("reason")
    print(f"{GREY}📍 Selected Route: {action}")
    print(f"📝 Reason: {reason}{RESET}\n")

    # Step 3 — the gate. Nothing is embedded, searched, or grounded past this point
    #          unless the role is permitted to use the chosen source.
    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        print(f"{RED}🚫 ACCESS DENIED{RESET} — role '{role}' may not query {source}.\n")
        return (
            f"🚫 Access denied: your role ('{role}') does not have permission to "
            f"query {source}."
        )

    print(f"{GREEN}✅ Access granted{RESET} — processing...\n")

    # Step 4 — identical to agentic_rag() from Section 5
    try:
        route_function = routes.get(action)
        if not route_function:
            return f"Unsupported action: {action}"
        if action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
            result = asyncio.run(route_function(user_query, action))
        else:
            result = route_function(user_query, action)
    except Exception as exec_err:
        result = f"Execution error: {exec_err}"

    print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
    print(f"{result}\n")
    return result

### Demo — same question, different roles

Each pair below sends the *identical* query as `alice` (engineer) and `bob`
(finance_analyst). The router makes the same decision both times — only the
permission check differs.


In [27]:
print("=" * 70)
print("1) alice (engineer) asks about the 10-K — FINANCE-ONLY → DENIED")
print("=" * 70)
secure_agentic_rag("alice", "what was uber revenue in 2021?")

1) alice (engineer) asks about the 10-K — FINANCE-ONLY → DENIED
👤 User: alice  (role: engineer)
❓ Query: what was uber revenue in 2021?

route_query: OPENAI (LLM) call to find the appropriate route for query: what was uber revenue in 2021?
route_query response: {'action': '10K_DOCUMENT_QUERY', 'reason': "Requests a company's annual financial revenue", 'answer': "Uber's 2021 revenue"}
📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Requests a company's annual financial revenue

🚫 ACCESS DENIED — role 'engineer' may not query 10-K financial filings.



"🚫 Access denied: your role ('engineer') does not have permission to query 10-K financial filings."

In [28]:
print("=" * 70)
print("2) bob (finance_analyst) asks the same question → ALLOWED")
print("=" * 70)
secure_agentic_rag("bob", "what was uber revenue in 2021?")

2) bob (finance_analyst) asks the same question → ALLOWED
👤 User: bob  (role: finance_analyst)
❓ Query: what was uber revenue in 2021?

route_query: OPENAI (LLM) call to find the appropriate route for query: what was uber revenue in 2021?
route_query response: {'action': '10K_DOCUMENT_QUERY', 'reason': 'Requests financial information from an annual report', 'answer': "Uber's 2021 revenue"}
📍 Selected Route: 10K_DOCUMENT_QUERY
📝 Reason: Requests financial information from an annual report

✅ Access granted — processing...

retrieve_and_response: get_text_embeddings for user_query
retrieve_and_response: get context (text hits and references) for the user_query embeddings from qdrant
retrieve_and_response: call rag_formatted_response and pass user_query and context
rag_formatted_response: OPENAI (llm) call with user_query and context
🤖 BOT RESPONSE:

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]



'Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]'

In [29]:
print("=" * 70)
print("3) bob (finance_analyst) asks for live web results — ENGINEER-ONLY → DENIED")
print("=" * 70)
secure_agentic_rag("bob", "List me down new LLMs in 2025")

3) bob (finance_analyst) asks for live web results — ENGINEER-ONLY → DENIED
👤 User: bob  (role: finance_analyst)
❓ Query: List me down new LLMs in 2025

route_query: OPENAI (LLM) call to find the appropriate route for query: List me down new LLMs in 2025
route_query response: {'action': 'INTERNET_QUERY', 'reason': 'Requests current information about newly released LLMs', 'answer': ''}
📍 Selected Route: INTERNET_QUERY
📝 Reason: Requests current information about newly released LLMs

🚫 ACCESS DENIED — role 'finance_analyst' may not query live internet search.



"🚫 Access denied: your role ('finance_analyst') does not have permission to query live internet search."

In [30]:
print("=" * 70)
print("4) alice (engineer) asks about OpenAI docs — SHARED → ALLOWED")
print("=" * 70)
secure_agentic_rag("alice", "best ways to build Agents")

4) alice (engineer) asks about OpenAI docs — SHARED → ALLOWED
👤 User: alice  (role: engineer)
❓ Query: best ways to build Agents

route_query: OPENAI (LLM) call to find the appropriate route for query: best ways to build Agents
route_query response: {'action': 'OPENAI_QUERY', 'reason': 'Building agents relates to OpenAI agent documentation', 'answer': 'Use the Agents SDK'}
📍 Selected Route: OPENAI_QUERY
📝 Reason: Building agents relates to OpenAI agent documentation

✅ Access granted — processing...

retrieve_and_response: get_text_embeddings for user_query
retrieve_and_response: get context (text hits and references) for the user_query embeddings from qdrant
retrieve_and_response: call rag_formatted_response and pass user_query and context
rag_formatted_response: OPENAI (llm) call with user_query and context
🤖 BOT RESPONSE:

# Best ways to build agents

## 1. Use an agent only where autonomy adds value

Agents are most useful for workflows that require multiple steps, decisions, tool 

'# Best ways to build agents\n\n## 1. Use an agent only where autonomy adds value\n\nAgents are most useful for workflows that require multiple steps, decisions, tool calls, or recovery from failures—for example, resolving support issues, researching a topic, booking services, or modifying code.\n\nAvoid using an agent for tasks that are deterministic or single-step, such as:\n\n- Simple chat responses\n- Sentiment classification\n- Fixed data transformations\n- Straightforward API workflows\n\nIn those cases, conventional software or a simple LLM call is usually more reliable and less expensive. [1]\n\n## 2. Start with the simplest agent architecture\n\nA practical agent has three core components:\n\n1. **Model** – handles reasoning and decision-making  \n2. **Tools** – APIs, functions, databases, search, or other external capabilities  \n3. **Instructions** – define behavior, workflow steps, constraints, and escalation rules  \n\nStart with one agent and a small set of well-defined t

**Where this is still weak — and how you'd harden it:**

- **The router runs before the check.** One LLM call is spent classifying a query the
  user may not be allowed to ask. That's cheap and leaks nothing, but you can do
  better: pass `allowed_sources(user_id)` into the router prompt so it only ever
  chooses from routes the role can reach, and deny anything that falls outside.
- **This gates whole sources, not chunks.** When one collection mixes content that
  different roles may only *partially* see, push the check into the vector store with
  **payload-based filters** (Qdrant supports this natively) so restricted chunks never
  enter the retrieved context in the first place. You'll do exactly this, at file
  granularity, in `003. Agentic Router_semantic_caching_rbac.ipynb`.
- **Caching and RBAC interact badly if you're careless.** A shared semantic cache
  keyed only on the question will happily serve `bob`'s finance answer to `alice`.
  Any cache sitting behind an access check must be partitioned by role (or by the
  permitted source set) — think about this before you add one.
- **Every check is an audit point.** Log who asked for what and whether it was
  allowed; that trail is what makes the system defensible in a regulated environment.


# Assignment

**Required:** Part 1 — sub-query division. This is the graded piece for this notebook.

**Bonus (optional, ungraded):** RBAC with a semantic cache. It extends Section 6 and is a
useful warm-up for **ARGUS**, where you build multi-source retrieval with a real caching
layer and have to report cost with and without the cache.

| | Task | Status | Builds on |
|---|---|---|---|
| **Part 1** | Sub-query division | **Required** | Sections 2 & 5 |
| **Bonus** | RBAC + semantic cache, without cross-role leakage | Optional | Section 6 |

**Deliverable:** this notebook, run end to end, with Part 1 implemented in the stub cell.
If you take the bonus, include it in the same notebook with the self-check passing.


---

## Part 1 — Sub-query division

Right now a compound question is treated as one search. Ask *"What was Uber's revenue
in 2021 and what was Lyft's in 2024?"* and the router picks a single route and fires a
single retrieval — so you get a partial answer, or a muddled one.

Your job: break compound queries into focused sub-queries, run each one through the
full agentic pipeline independently, then compose a single coherent answer.

**Requirements**

1. Write `agentic_rag_multi(user_query)` that:
   - calls `sub_queries()` to split the query (reference implementation below),
   - **routes each sub-query separately** — they may legitimately land on different
     sources (one on `10K_DOCUMENT_QUERY`, another on `INTERNET_QUERY`),
   - collects the per-sub-query answers and synthesises **one** final response,
   - preserves citations from each sub-answer in the composed output.
2. Handle the single-question case without regression — one question in, one route,
   no extra LLM calls beyond the split.
3. Parse the model's JSON defensively. `sub_queries()` returns a *string*; it can come
   back wrapped in prose or a code fence. Don't let a malformed split crash the agent —
   fall back to treating the input as one query.

**Check yourself against these**

| Query | Expected behaviour |
|---|---|
| `"what was uber revenue in 2021?"` | 1 sub-query, 1 route, same as `agentic_rag()` |
| `"what was lyft revenue in 2021 and what was uber revenue in 2021"` | 2 sub-queries, both `10K_DOCUMENT_QUERY` |
| `"what was uber's 2021 revenue and what are the newest LLMs?"` | 2 sub-queries, **different** routes |

**Stretch:** run the sub-queries concurrently with `asyncio.gather` instead of in
sequence, and compare wall-clock time.


In [32]:
#Reference Code for sub query division (For Guidance Only)

def sub_queries(user_query):
  sub_queries_prompt= f"""
  You are a query router. If the input contains multiple distinct questions, break it into sub-questions. Otherwise, keep it as one. Return a JSON object like:

  {{
      "subQuestions": ["..."]
  }}


  Query: "{user_query}"
  Output:
  """
  print(f"sub_queries: OPENAI (LLM) call to break the query to sub-queries")
  response = openaiclient.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": sub_queries_prompt},
        ]
    )
  
  sub_query_string = response.choices[0].message.content
  print(f"sub_queries response: {sub_query_string}")
  return response.choices[0].message.content


In [33]:
print(sub_queries("what was Uber's revenue in 2021"))

sub_queries: OPENAI (LLM) call to break the query to sub-queries
sub_queries response: {"subQuestions":["What was Uber's revenue in 2021?"]}
{"subQuestions":["What was Uber's revenue in 2021?"]}


In [34]:
print(sub_queries("what was lyft revenue in 2021 and what was uber revenue in 2021"))

sub_queries: OPENAI (LLM) call to break the query to sub-queries
sub_queries response: {"subQuestions":["What was Lyft's revenue in 2021?","What was Uber's revenue in 2021?"]}
{"subQuestions":["What was Lyft's revenue in 2021?","What was Uber's revenue in 2021?"]}


In [35]:
# ── Part 1: your implementation ──────────────────────────────────────────────
import json
import re

# Requirement 3: Parse the model's JSON defensively. 
#   sub_queries() returns a string; 
#   it can come back wrapped in prose or a code fence. 
#   Don't let a malformed split crash the agent 
#   — fall back to treating the input as one query.
# Input:    raw_sub_query (string)  - this is usually the output of sub_queries()
#           user_query (string)     - use this to return in case of exception
# Output: a list of sanitized sub-query strings
# With help from Claude Code
def parse_sub_queries(raw: str, user_query: str ) -> list[str]:
    try:
        # 1. remove leading and trailing empty spaces, tabs, new line characters
        text = raw.strip()

        # 2. remove code fence (``` used in markdown)
        # ```json ["abc", "def"]``` => ["abc", "def"]
        # uses regular expression to check for presense of fence
        # if fence is present, extract the string enclosed between the fence and strip empty space, tab, newlines
        fence = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
        if fence:
            text = fence.group(1).strip()

        # the string extracted after removing fence may have other prose around json data
        # if there was no fence, the original raw data may have other prose around json data
        # Extract only the json content - find [ and ], then extract data between them
        start, end = text.find("["), text.find("]")
        if start == -1 or end <= start:
            raise ValueError("no JSON array found")
        data = json.loads(text[start:end+1])

        # Validate that data is in a list format
        if not isinstance(data, list):
            raise ValueError("not a list")

        # Validate each of the sub-query available in data[]
        # 1 liner -> queries = [q.strip() for q in data if isinstance(q, str) and q.strip()]
        # or the following for better understanding
        queries = []
        for item in data:
            # ignore anything that is not a string
            if not isinstance(item, str):
                continue

            # trim spaces, tabs, new lines from both ends
            cleaned = item.strip()

            # skip empty strings
            if cleaned == "":
                continue

            queries.append(cleaned)

        if not queries:
            raise ValueError("no usable sub-queries")         

        return queries
    
    except (ValueError, json.JSONDecodeError):
        # Bad raw data. treat input as a single query and return a list with that query
        return [user_query]
    

In [36]:
# Requirement:
#   collect the per-sub-query answers and synthesise one final response
#   preserve citations from each sub-answer in the composed output.
# Input:  user_query (string)   - the user's original (possibly compound) question
#         resultSet (list of dict) - each dict is {"question", "action", "answer"}
#                                    for one sub-query, in the order they were answered
# Output: a single composed answer string. If the synthesis call itself fails,
#         falls back to a concatenation of the individual answers so nothing is lost.
def synthesize_answers(user_query: str, resultSet: list[dict]) -> str:

    # convert into a better list for printing
    sections = "\n\n".join(
        f"Sub-question: {r['question']}\n"
        f"Source: {r['action']}\n"
        f"Answer:\n{r['answer']}"
        for r in resultSet
    )

    synthesis_prompt = f"""
    Compose one single coherent answer to a user's original question from the answers already gathered for 
    each of its sub-questions.

    Original question: "{user_query}"

    Sub-answers:
    {sections}

    Write ONE coherent answer that addresses every sub-question. Keep every
    citation marker exactly as it appears in each sub-answer (e.g. [1], [2],
    "Source: ..." lines) — do not renumber or merge citation markers across
    sub-answers, since each was numbered independently. 
    Do not invent information. Any answer should be present in the sub-answers.
    """

    try:
        print(f"synthesize_answers: OPENAI (LLM) call to generate a coherent answer")
        response = openaiclient.chat.completions.create(
            model="gpt-5.6-luna",
            messages=[{"role": "system", "content": synthesis_prompt}],
        )
        return response.choices[0].message.content
    except Exception as synth_err:
        # Don't lose the individual answers just because composition failed
        fallback = "\n\n".join(f"**{r['question']}**\n{r['answer']}" for r in resultSet)
        return f"(synthesis failed: {synth_err})\n\n{fallback}"

In [37]:
# Assignment - update agentic_rag_multi
def agentic_rag_multi(user_query: str):
    """
    Split a compound query, run each sub-query through the agentic pipeline,
    and synthesise one final answer.

    Args:
        user_query (str): Possibly compound question.

    Returns:
        str: A single composed answer covering every sub-question.
    """

    #  Terminal color codes to make the printed output easier to read and visually structured
    CYAN = "\033[96m"
    GREY = "\033[90m"
    BOLD = "\033[1m"
    RESET = "\033[0m"
    
    print(f"agentic_rag_multi: processing user query: {user_query}")

    if not user_query:
        return "user_query is empty"

    # 1. Split the user query into sub-queries
    try:
        raw_sub_queries = sub_queries(user_query)
    except Exception as sub_queries_err:
        print(f"Exception getting sub_queries: {sub_queries_err}")
        return 

    try:
        sanitized_sub_queries = parse_sub_queries (raw_sub_queries, user_query)
    except Exception as parse_sub_queries_err:
        print(f"Exception getting sub_queries: {parse_sub_queries_err}")
        return
    
    resultSet = []
    # 2. Route + answer each sub-query (reuse route_query + routes, as agentic_rag does)
    try:
        # for each sub query
        #   identify the route
        #   call appropriate route to get an answer
        #   append the answer to a list of sub_question, route, reason, answer data structure
        for sub_query in sanitized_sub_queries:
            try:
                route = route_query(sub_query)
                route_action = route.get("action")  # e.g., "OPENAI_QUERY"
            except Exception as route_err:
                # If something goes wrong while classifying the query, show an error message
                # and continue with the next sub-query
                print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
                print(f"Routing error: {route_err}\n")
                resultSet.append({
                    "question": sub_query,
                    "action": None,
                    "answer": f"Routing error: {route_err}",
                })
                continue            

            try:
                route_function = routes.get(route_action)  # Find the function to use for this route
                if route_function:
                    if route_action in ["OPENAI_QUERY", "10K_DOCUMENT_QUERY"]:
                        # Use asyncio.run for async functions, nest_asyncio will handle nested loops
                        result = asyncio.run(route_function(sub_query, route_action))
                    else:
                        # Otherwise, call it directly (e.g., get_internet_content is synchronous)
                        result = route_function(sub_query, route_action)
                else:
                    result = f"Unsupported action: {route_action}"  # Catch unknown routing types
            except Exception as exec_err:
                result = f"Execution error: {exec_err}"  # Handle failure in the chosen route function

            resultSet.append({
                "question": sub_query,
                "action": route_action,
                "answer": result,
            })
            print(f"{BOLD}{CYAN}🤖 BOT RESPONSE:{RESET}\n")
            print(f"{result}\n")
            
    except Exception as e:
        return f"Exception processing subqueries: {e}"

    # 3. Compose one final answer from the parts, keeping citations
    if not resultSet:
        return "This user query produced no answers"

    # Requirement: Handle the single-question case without regression 
    #   — one question in, one route, no extra LLM calls beyond the split.
    if len(resultSet) == 1:
        return resultSet[0]["answer"]    

    # Requirement: 
    # collects the per-sub-query answers and synthesises one final response,
    # preserves citations from each sub-answer in the composed output.
    coherent_answer = synthesize_answers(user_query, resultSet)

    return coherent_answer

# Test cases from the table above
# agentic_rag_multi("what was uber revenue in 2021?")
# agentic_rag_multi("what was lyft revenue in 2021 and what was uber revenue in 2021")
# agentic_rag_multi("what was uber's 2021 revenue and what are the newest LLMs?")

In [38]:
agentic_rag_multi("")

agentic_rag_multi: processing user query: 


'user_query is empty'

In [39]:
agentic_rag_multi("what was Uber's revenue in 2021")

agentic_rag_multi: processing user query: what was Uber's revenue in 2021
sub_queries: OPENAI (LLM) call to break the query to sub-queries
sub_queries response: {"subQuestions":["What was Uber's revenue in 2021?"]}
route_query: OPENAI (LLM) call to find the appropriate route for query: What was Uber's revenue in 2021?
route_query response: {'action': '10K_DOCUMENT_QUERY', 'reason': "Requests historical financial data from Uber's annual report", 'answer': "Uber's 2021 revenue"}
retrieve_and_response: get_text_embeddings for user_query
retrieve_and_response: get context (text hits and references) for the user_query embeddings from qdrant
retrieve_and_response: call rag_formatted_response and pass user_query and context
rag_formatted_response: OPENAI (llm) call with user_query and context
🤖 BOT RESPONSE:

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**).[1]



'Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**).[1]'

In [40]:
agentic_rag_multi("what was lyft revenue in 2021 and what was uber revenue in 2021")

agentic_rag_multi: processing user query: what was lyft revenue in 2021 and what was uber revenue in 2021
sub_queries: OPENAI (LLM) call to break the query to sub-queries
sub_queries response: {"subQuestions":["What was Lyft's revenue in 2021?","What was Uber's revenue in 2021?"]}
route_query: OPENAI (LLM) call to find the appropriate route for query: What was Lyft's revenue in 2021?
route_query response: {'action': '10K_DOCUMENT_QUERY', 'reason': "Requests financial information from Lyft's annual report", 'answer': "Lyft's 2021 revenue"}
retrieve_and_response: get_text_embeddings for user_query
retrieve_and_response: get context (text hits and references) for the user_query embeddings from qdrant
retrieve_and_response: call rag_formatted_response and pass user_query and context
rag_formatted_response: OPENAI (llm) call with user_query and context
🤖 BOT RESPONSE:

Lyft’s revenue in 2021 was **$3.208 billion** (approximately **$3,208,323,000**). [1]

route_query: OPENAI (LLM) call to fi

'In 2021:\n\n- **Lyft’s revenue:** **$3.208 billion** (approximately **$3,208,323,000**). [1]\n- **Uber’s revenue:** **$17.455 billion** (approximately **$17.5 billion**). [1]'

In [41]:
agentic_rag_multi("what was uber's 2021 revenue and what are the newest LLMs?")

agentic_rag_multi: processing user query: what was uber's 2021 revenue and what are the newest LLMs?
sub_queries: OPENAI (LLM) call to break the query to sub-queries
sub_queries response: {"subQuestions":["What was Uber's revenue in 2021?","What are the newest large language models (LLMs)?"]}
route_query: OPENAI (LLM) call to find the appropriate route for query: What was Uber's revenue in 2021?
route_query response: {'action': '10K_DOCUMENT_QUERY', 'reason': "Query asks for financial data from Uber's annual report.", 'answer': '$17.46 billion'}
retrieve_and_response: get_text_embeddings for user_query
retrieve_and_response: get context (text hits and references) for the user_query embeddings from qdrant
retrieve_and_response: call rag_formatted_response and pass user_query and context
rag_formatted_response: OPENAI (llm) call with user_query and context
🤖 BOT RESPONSE:

Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]

route_query: OPENAI (LLM) cal

'Uber’s revenue in 2021 was **$17.455 billion** (approximately **$17.5 billion**). [1]\n\nSome of the newest and most prominent LLMs identified in the available sources include **ChatGPT, Claude, Gemini, DeepSeek, Grok, Mistral, and Llama**. [1] [4] The broader LLM landscape also includes models such as **GPT-3, GPT-4, LaMDA, BLOOM, and LLaMA**. [2]\n\n[1] Best LLMs in 2026: I Reviewed 7 Large Language Models  \nI evaluated 44+ tools to find the 7 best large language models (LLMs) software in 2026. These include ChatGPT, Claude, Gemini, Deepseek, Grok, ...  \nSource: https://www.g2.com/articles/best-llms\n\n[2] What are large language models (LLMs)  \nSuch renowned models like GPT-3, GPT-4, LaMDA, BLOOM, LLaMA, are behind the scenes in many applications we interact with daily, such , and ...  \nSource: https://www.superannotate.com/blog/llm-overview\n\n[3] 7 new large language models released in the last 30 days ...  \n>7 new large language models released in the last 30 days to Apr/20

---

## Bonus (optional) — RBAC with a semantic cache

*Not graded on its own — but this is rehearsal for graded work. **ARGUS**, the full-stack
assignment for this module, requires a real caching layer and a cost panel showing spend per
100 sources with and without it. Build the cache here, on a system you already understand,
and you'll be extending it in ARGUS rather than meeting it for the first time under a deadline.*

> **Come back to this after notebooks 002 and 003** — `002. Semantic Caching.ipynb`
> for how a FAISS cache works, and `003. Agentic Router_semantic_caching_rbac.ipynb` for `SemanticCaching`
> in `rag_helpers.py` and the file-level RBAC gate you'll be extending. You can read the spec now;
> you'll have every piece you need once those two are done.

Section 6 gates access by role. A semantic cache makes repeat questions near-instant.
Put them together naively and you build a data leak: the cache is keyed on the
*question*, so once `bob` (finance_analyst) asks about Uber's revenue, `alice`
(engineer) asks the same thing, hits the cache, and is handed finance data the RBAC
gate was supposed to deny her — without a single retrieval ever running.

Your job: add caching to `secure_agentic_rag()` so that repeat questions are fast
**and** no answer ever crosses a permission boundary.

**Requirements**

1. Build a role-aware cache. Two viable designs — pick one and justify it in a comment:
   - **Partitioned:** a separate FAISS index (or a namespace) per role, so a lookup
     can only ever see entries its own role produced.
   - **Tagged:** one index, but each entry stores the role (or the permitted source
     set) that produced it, and a hit is only honoured when it matches the caller.
2. Write `secure_agentic_rag_cached(user_id, user_query)` with this order of
   operations — it matters:
   ```
   unknown user      → DENIED   (no embedding, no cache, no LLM)
   route the query   → which source does this need?
   role not allowed  → DENIED   (still no cache lookup — a denial must not be cacheable)
   cache lookup      → HIT  → return stored answer
                     → MISS → run the pipeline, store, return
   ```
3. Return a dict, not a bare string, so the self-check can verify behaviour:
   `{"answer": str, "status": "HIT" | "MISS" | "DENIED", "role": str | None}`
4. Never cache a denial, and never cache a time-sensitive query. Reuse the
   `is_time_sensitive()` idea from `003. Agentic Router_semantic_caching_rbac.ipynb` — a stock
   price cached for an hour is a wrong answer served fast.
5. Make the leak test below pass.

**Hints**

- `SemanticCaching` in `rag_helpers.py` is a working FAISS cache — read it first.
  Partitioning it is mostly a matter of what you key and what you search.
- Think about what happens when a user's role *changes*. Should their old cache
  entries still be reachable? Write a sentence on your answer.
- Two roles share `OPENAI_QUERY`. A strictly per-role cache re-computes that answer
  once per role — correct, but wasteful. Caching per *permitted source* instead of per
  role fixes it. Trade-off worth a comment.

**Stretch:** log every request as an audit record — user, role, query, route, decision,
cache status, latency — and print a small table at the end. That table is what you'd
hand an auditor.


In [42]:
# Utility class to check time sensitivity
# Helps with Requirement 4
class TimeSensitivity:
    # copied from SemanticCaching rag_helpers.py
    TIME_SENSITIVE_KEYWORDS = [
        "today", "tonight", "now", "currently", "current",
        "latest", "recent", "recently", "right now", "at the moment",
        "at present", "as of now", "this week", "this month", "this year",
        "this quarter", "this season", "this morning", "this afternoon",
        "this evening", "this weekend", "yesterday", "tomorrow",
        "last week", "last month", "last year", "upcoming", "live",
        "breaking", "just happened", "what time", "what day", "what date",
        "happening now", "events today", "news today", "news this week",
        "stock price", "share price", "weather", "forecast", "temperature",
        "real-time", "realtime", "schedule today",
    ]

    # copied from SemanticCaching rag_helpers.py
    @classmethod
    def is_time_sensitive(cls, question: str) -> bool:
        """
        Return True if the question contains temporal indicators whose
        answers change over time and should never be served from cache.

        Examples that return True  → bypass cache, always fetch live:
            'What is the current stock price of AAPL?'
            'What are the latest AI news this week?'
            'Are there any AWS outages right now?'
    
        Examples that return False → safe to cache:
            'What was Uber revenue in 2021?'
            'How do OpenAI Agents work?'
            'What is machine learning?'
        """
        q = question.lower()
        return any(kw in q for kw in cls.TIME_SENSITIVE_KEYWORDS)

# User Role validation checks


In [43]:
# creating a separate function so that it can be called in different sections
ASYNC_ACTIONS = {"OPENAI_QUERY", "10K_DOCUMENT_QUERY"}

def execute_query(query: str, action: str) -> str:
    """
    Execute query against the specified action (one of ASYNC_ACTIONS) and return the answer as a string    
    """
    # check if the action is a defined value
    route_function = routes.get(action)
    if route_function is None:
        return f"Unsupported action: {action}"

    if action in ASYNC_ACTIONS:
        # action is either OPENAI_QUERY or 10K_DOCUMENT_QUERY
        # asyncio.run for async routes and nest_asyncio handles nested loops
        return asyncio.run(route_function(query, action))
    else:
        # INTERNET_QUERY
        return route_function(query, action)

In [68]:
import time
from datetime import datetime

class AuditLog:
    def __init__(self):
        self.records = []

    def add(self, record: dict):
        self.records.append(record)

    def print_table(self):
        ...

audit_log = AuditLog()

In [63]:
# ── Bonus: your implementation ───────────────────────────────────────────────
# Reuse USERS / ROLE_PERMISSIONS / has_access / SOURCE_LABELS from Section 6.

# 1. Role aware-cache   -> implement in class RoleAwareSemanticCache
# 2. secure_agentic_rag_cached(user_id, user_query) -> implement in def secure_agentic_rag_cached
# 3. Return a dict      ->  implement in secure_agentic_rag_cached
# 4. Never cache a denial, never cache a time-sensitive query
#    reuse is_time_sensitive() from Agentic_Router_semantic_caching)rbac.ipynb. 
#    is_time_sensitive() is implemented in a separate class class TimeSensitivity   

import numpy as np

class RoleAwareSemanticCache:
    """
    A semantic cache that cannot serve an answer across a permission boundary.

    Design choice (partitioned vs. tagged): partitioned - a separate cache data structure for each data source because,
        each index can be audited separately and the data store/cache optimized independently depending on usage.
        Each cache is accessible only if the underlying data source is accessible to a role.
        A user can access a cache only if the user has a role that can access the underlying data source

    - every users needs to be associated with a role
    - when there is no role defined for a user, that user's query is DENIED    
    - each role maintains a corresponding data structure that the role can access
    - when multiple roles can access the same FAISS index, the index is not recomputed per role. they share the same index
    """

    # initialize default threshold
    # 
    def __init__(self, threshold: float = 0.2):
        self.threshold = threshold
        self.datasources = {}
        # raise NotImplementedError("Bonus — implement me")

    @staticmethod
    def _normalize(vector) -> np.ndarray:
        # unit length, so a dot product equals cosine similarity
        v = np.asarray(vector, dtype="float32").ravel()
        return v / np.linalg.norm(v)


    # 'user_id' needs to have access to 'action' based on user's role
    def _require_access(self, user_id: str, action: str):
        if not user_id:
            raise Exception("user_id is empty")
        if not action:
            raise Exception("action is empty")
        if not has_access(user_id, action):
            raise Exception("Permission denied")

    def check(self, user_id: str, question: str, action: str):
        """Return (hit: bool, answer: str | None, embedding, similarity | None)."""
        # validation checks
        self._require_access(user_id, action)

        embedding = self._normalize(get_text_embeddings(question))
        items_for_action = self.datasources.get(action)
        # Not cached yet
        if not items_for_action:
            return False, None, embedding, None

        # find similarity
        matrix = np.vstack([item["vector"] for item in items_for_action])
        scores = matrix @ embedding         # cosine similarity per stored item
        best = int(np.argmax(scores))
        similarity = float(scores[best])

        if (1.0 - similarity) <= self.threshold:
            return True, items_for_action[best]["answer"], embedding, similarity
        else:
            return False, None, embedding, similarity
        

    # def add(self, user_id: str, question: str, answer: str, embedding):
    def add (self, user_id: str, question: str, answer: str, embedding, action: str):
        """Store an answer scoped to this user's permissions."""
        
        # validation checks
        self._require_access(user_id, action)
        if embedding is None or np.size(embedding) == 0:
            raise Exception("embedding is empty")
    
        # add this data to a list indexed by datasource names
        normalized_embedding = self._normalize(embedding)
        item = {
            "vector": normalized_embedding,
            "question": question,
            "answer": answer,
        }

        if action not in self.datasources:
            self.datasources[action] = []

        self.datasources[action].append(item)


def _secure_agentic_rag_cached(user_id: str, user_query: str, cache, record) -> dict:
    """
    RBAC-gated agentic RAG with a role-aware semantic cache.

    Order: identity → route → permission → cache → pipeline.

    Returns:
        dict: {"answer": str, "status": "HIT" | "MISS" | "DENIED", "role": str | None}
    """
    # Steps
    # 1. get the role for this user
    # 2. identify the route for this query
    # 3. [validation] check if the role has permissions to access this route
    # 4. if the query is time sensitive -> return search result without caching
    # 5. check if the query is cached in FAISS for this route
    # 5a. cache HIT -> return the cached answer
    # 5b. cache MISS -> perform search, cache it and then return the results

    # Step 1
    if not user_id:
        return {"answer": f"🚫 ACCESS DENIED - invalid user_id (EMPTY)", "status": "DENIED", "role": None}

    role = USERS.get(user_id)
    if role is None:
        record["role"] = "None"
        return {"answer": f"🚫 ACCESS DENIED - no role for user_id {user_id}", "status": "DENIED", "role": None}

    # audit
    record["role"] = role

    # Step 2
    try:
        decision = route_query(user_query)
        record["route"] = decision      # audit
    except Exception as route_err:
        record["route"] = "Exception-Unknown"
        record["decision"] = "DENIED"
        return {"answer": f"🚫 Routing error - {route_err}", "status": "DENIED", "role": None}
 
    # Step 3
    action = decision.get("action")
    if not has_access(user_id, action):
        source = SOURCE_LABELS.get(action, action)
        record["decision"] = "DENIED"
        return {"answer": f"🚫 ACCESS DENIED — '{user_id}' has '{role}' and role may not query {source}", "status": "DENIED", "role": role}
    else:
        record["decision"] = "Approved"
        print(f"✅ Access granted — for user {user_id} to access the data source, processing...\n")


    record["cache"] = "MISS"        # default value. change to HIT only when a true HIT
    # Step 4
    if TimeSensitivity.is_time_sensitive(user_query):
        print(f"this query is time sensitive, skipping cache")
        # Skip cache entirely. Call data source and return the answer
        try:
            # return the value of running the query against specific action/source
            result = execute_query(user_query, action)
            return {"answer": result, "status": "MISS", "role": role}
        except Exception as query_err:
            return {"answer": f"Exception with querying the time sensitive request", "status": "MISS", "role": role}
    else:
        print(f"this query is not time sensitive, go through caching system")
        if cache is None:
            print(f"cache is unreachable. assuming it is a MISS and going through data source")
            try:
                result = execute_query(user_query, action)
                return {"answer": result, "status": "MISS", "role": role}
            except Exception as query_err:
                return {"answer": f"Exception with querying the data sources", "status": "MISS", "role": role}
        else:
            # RoleAwareSemanticCache.check returns (hit: bool, answer: str | None, embedding, similarity | None)
            try:
                hit, answer, embedding, similarity = cache.check(user_id, user_query, action)
                if hit:
                    record["cache"] = "HIT" 
                    return {"answer": answer, "status": "HIT", "role": role}
                else:
                    try:
                        result = execute_query(user_query, action)
                        # add result to cache
                        try:
                            cache.add(user_id, user_query, result, embedding, action)
                        except Exception as cache_add:
                            print(f"Error adding answer to the cache: {cache_add}")    

                        return {"answer": result, "status": "MISS", "role": role}
                    except Exception as query_err:
                        return {"answer": f"Exception with querying the data sources", "status": "MISS", "role": role} 
            except Exception as cache_err:
                return {"answer": f"Exception with querying the cache: {cache_err}", "status": "MISS", "role": role}
    

In [69]:
import time
from datetime import datetime

def secure_agentic_rag_cached(user_id: str, user_query: str, cache) -> dict:
    record = {
        "time": datetime.now().strftime("%H:%M:%S"),
        "user": user_id or "(empty)",
        "role": None,
        "query": user_query,
        "route": None,
        "decision": None,
        "cache": None
    }
    start = time.perf_counter()
    try:
        return _secure_agentic_rag_cached(user_id, user_query, cache, record)
    finally:
        record["latency_ms"] = round((time.perf_counter() - start) * 1000)
        audit_log.add(record)


In [45]:
# empty user check
secure_agentic_rag_cached("", "what is 10-K", RoleAwareSemanticCache())

{'answer': '🚫 ACCESS DENIED - invalid user_id (EMPTY)',
 'status': 'DENIED',
 'role': None}

In [46]:
secure_agentic_rag_cached("unknown_user", "what is 10-K", RoleAwareSemanticCache())

{'answer': '🚫 ACCESS DENIED - no role for user_id unknown_user',
 'status': 'DENIED',
 'role': None}

In [50]:
# user, role -> no permission (alice may not read financial information)
secure_agentic_rag_cached("alice", "what was Uber revenue in 2021", RoleAwareSemanticCache())

route_query: OPENAI (LLM) call to find the appropriate route for query: what was Uber revenue in 2021
route_query response: {'action': '10K_DOCUMENT_QUERY', 'reason': "Requests Uber's historical annual revenue", 'answer': 'Uber revenue was $17.5 billion'}


{'answer': "🚫 ACCESS DENIED — 'alice' has 'engineer' and role may not query 10-K financial filings",
 'status': 'DENIED',
 'role': 'engineer'}

In [56]:
# user, role -> permission (alice can read OpenAI documents)
secure_agentic_rag_cached("alice", "How to create AI agents", RoleAwareSemanticCache())

route_query: OPENAI (LLM) call to find the appropriate route for query: How to create AI agents
route_query response: {'action': 'OPENAI_QUERY', 'reason': 'Creating AI agents relates to OpenAI Agents documentation', 'answer': 'Use the Agents SDK'}
✅ Access granted — for user alice to access the data source, processing...

this query is not time sensitive, go through caching system
retrieve_and_response: get_text_embeddings for user_query
retrieve_and_response: get context (text hits and references) for the user_query embeddings from qdrant
retrieve_and_response: call rag_formatted_response and pass user_query and context
rag_formatted_response: OPENAI (llm) call with user_query and context


{'answer': '## How to create an AI agent\n\nAn AI agent is an application that uses an LLM to independently manage a workflow, make decisions, use tools, recognize when a task is complete, and stop or hand control back to the user when necessary.[2]\n\n### 1. Define the task\n\nChoose a workflow where the model must decide what to do next, rather than simply generate a single response. Examples include:\n\n- Answering questions using external data\n- Checking the weather\n- Processing customer-service requests\n- Booking reservations\n- Generating reports\n- Translating content\n\nA simple chatbot or one-off LLM call is not necessarily an agent; the LLM must participate in controlling the workflow.[2]\n\n### 2. Define the three core components\n\nEvery basic agent consists of:\n\n1. **Model** – the LLM responsible for reasoning and decision-making  \n2. **Tools** – functions, APIs, databases, or services the agent can call  \n3. **Instructions** – behavioral rules, objectives, and safe

In [57]:
# user, role -> permission (alice can read OpenAI documents)
cache_temp = RoleAwareSemanticCache()
secure_agentic_rag_cached("alice", "How to create AI agents", cache_temp)

print(f"the previous call would be a cache MISS. next call should be a HIT")
secure_agentic_rag_cached("alice", "How to create AI agents", cache_temp)

route_query: OPENAI (LLM) call to find the appropriate route for query: How to create AI agents
route_query response: {'action': 'OPENAI_QUERY', 'reason': "Creating AI agents relates to OpenAI's official Agents documentation", 'answer': 'Use the Agents SDK'}
✅ Access granted — for user alice to access the data source, processing...

this query is not time sensitive, go through caching system
retrieve_and_response: get_text_embeddings for user_query
retrieve_and_response: get context (text hits and references) for the user_query embeddings from qdrant
retrieve_and_response: call rag_formatted_response and pass user_query and context
rag_formatted_response: OPENAI (llm) call with user_query and context
the previous call would be a cache MISS. next call should be a HIT
route_query: OPENAI (LLM) call to find the appropriate route for query: How to create AI agents
route_query response: {'action': 'OPENAI_QUERY', 'reason': 'AI agents are covered by OpenAI agent documentation', 'answer': 'Us

{'answer': '## How to create an AI agent\n\nAn AI agent is a system that uses an LLM to independently complete a task by reasoning through a workflow, deciding what to do next, using tools, and recognizing when the task is complete.[1]\n\n### 1. Define the agent’s goal\n\nChoose a task that involves multiple steps or decisions, such as:\n\n- Answering customer-support questions\n- Checking weather or inventory\n- Booking appointments\n- Generating reports\n- Translating content\n- Processing refunds\n\nSimple single-turn chatbots or classifiers are not necessarily agents; an agent uses the LLM to control workflow execution.[1]\n\n### 2. Choose the three core components\n\nEvery agent generally consists of:\n\n1. **Model** — the LLM responsible for reasoning and decision-making  \n2. **Tools** — functions, databases, or APIs the agent can call  \n3. **Instructions** — rules that define the agent’s behavior and constraints[1]\n\nFor example:\n\n```python\nfrom agents import Agent, Runner

In [58]:
# Step 4: Time sensitive check - time sensitive "current"
secure_agentic_rag_cached("alice", "what is the current status of AI Agents", RoleAwareSemanticCache())

route_query: OPENAI (LLM) call to find the appropriate route for query: what is the current status of AI Agents
route_query response: {'action': 'INTERNET_QUERY', 'reason': 'Broad, current AI industry status requires wider research', 'answer': ''}
✅ Access granted — for user alice to access the data source, processing...

this query is time sensitive, skipping cache
Getting your response from the internet 🌐 ...
calling INTERNET search using SERPAPI for: what is the current status of AI Agents


{'answer': '[1] AI agents: Here is what happened in 2025 and the ...\n    In 2025, AI agents shifted from theory to practice, the recent change is their expanding capacity to act, using tools, calling APIs, the stage ...\n    Source: https://www.business-standard.com/technology/tech-news/ai-agents-tech-recap-what-happened-2025-challenges-in-2026-126011500209_1.html\n\n[2] “Are AI agents ready for the enterprise?”\n    Deepak Singh, VP of Developer Agents and Experiences at AWS, helps Ryan break down the hype around agentic AI in software development.\n    Source: https://stackoverflow.blog/2025/03/28/are-ai-agents-ready-for-the-enterprise/\n\n[3] What are Different Types of AI Agents With Examples in 2026\n    According to industry reports, the global AI agent market is projected to grow from $5.1 billion in 2024 to $52.6 billion by 2030,\n    Source: https://interviewkickstart.com/blogs/articles/different-types-of-ai-agents\n\n[4] Context Agents: How They Work and Why They Fail in 202

In [59]:
# Steo 4: Non-time sensitive
secure_agentic_rag_cached("bob", "what is the price of Uber in 2021", RoleAwareSemanticCache())

route_query: OPENAI (LLM) call to find the appropriate route for query: what is the price of Uber in 2021
route_query response: {'action': '10K_DOCUMENT_QUERY', 'reason': "Uber's 2021 financial information is found in annual reports.", 'answer': "Check Uber's 2021 annual report"}
✅ Access granted — for user bob to access the data source, processing...

this query is not time sensitive, go through caching system
retrieve_and_response: get_text_embeddings for user_query
retrieve_and_response: get context (text hits and references) for the user_query embeddings from qdrant
retrieve_and_response: call rag_formatted_response and pass user_query and context
rag_formatted_response: OPENAI (llm) call with user_query and context


{'answer': 'The provided context does not state Uber’s stock price in 2021. It reports **2021 revenue of $17.455 billion** and a **net loss per share of $0.26 basic ($0.29 diluted)**.[1]',
 'status': 'MISS',
 'role': 'finance_analyst'}

In [74]:
# ── Bonus: self-check — this must pass ───────────────────────────────────────
# It asserts behaviour, not wording, so your answer text can be anything.
# DO NOT CHANGE ANYTHING IN THIS CELL

def run_self_check():
    cache = RoleAwareSemanticCache()
    q_fin = "what was uber revenue in 2021?"
    q_doc = "how do I build an agent with the OpenAI Agents SDK?"

    # 1. bob may read financials — first ask is a MISS
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "MISS", f"expected MISS, got {r['status']}"

    # 2. bob asks again — served from cache
    r = secure_agentic_rag_cached("bob", q_fin, cache)
    assert r["status"] == "HIT", f"expected HIT, got {r['status']}"

    # 3. THE LEAK TEST — alice must be denied, never served bob's cached answer
    r = secure_agentic_rag_cached("alice", q_fin, cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} on finance data"

    # 4. a near-paraphrase must also be denied, not semantically matched into bob's rows
    r = secure_agentic_rag_cached("alice", "how much revenue did Uber make in 2021?", cache)
    assert r["status"] == "DENIED", f"LEAK: alice got {r['status']} via paraphrase"

    # 5. unknown users are rejected outright
    r = secure_agentic_rag_cached("carol", q_doc, cache)
    assert r["status"] == "DENIED", f"expected DENIED for unknown user, got {r['status']}"

    # 6. a shared source still caches normally within a role
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "MISS"
    assert secure_agentic_rag_cached("alice", q_doc, cache)["status"] == "HIT"

    print("✅ All checks passed — cache is fast and does not leak across roles.")


# run_self_check()

In [75]:
run_self_check()

route_query: OPENAI (LLM) call to find the appropriate route for query: what was uber revenue in 2021?
route_query response: {'action': '10K_DOCUMENT_QUERY', 'reason': 'Requests financial information from an annual report', 'answer': 'Uber revenue was $17.5 billion'}
✅ Access granted — for user bob to access the data source, processing...

this query is not time sensitive, go through caching system
retrieve_and_response: get_text_embeddings for user_query
retrieve_and_response: get context (text hits and references) for the user_query embeddings from qdrant
retrieve_and_response: call rag_formatted_response and pass user_query and context
rag_formatted_response: OPENAI (llm) call with user_query and context
route_query: OPENAI (LLM) call to find the appropriate route for query: what was uber revenue in 2021?
route_query response: {'action': '10K_DOCUMENT_QUERY', 'reason': "Revenue from a company's annual report", 'answer': 'Uber reported $17.5 billion'}
✅ Access granted — for user bob 

In [76]:
%pip install pandas
import pandas as pd

pd.DataFrame(audit_log.records)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


,time,user,role,query,route,decision,cache,latency_ms
0,20:42:38,bob,finance_analyst,what was uber revenue in 2021?,"{'action': '10K_DOCUMENT_QUERY', 'reason': 'Qu...",Approved,MISS,3814
1,20:42:42,bob,finance_analyst,what was uber revenue in 2021?,"{'action': '10K_DOCUMENT_QUERY', 'reason': 'Ub...",Approved,HIT,1596
2,20:42:44,alice,engineer,what was uber revenue in 2021?,"{'action': '10K_DOCUMENT_QUERY', 'reason': 'As...",DENIED,NaN,861
3,20:42:45,alice,engineer,how much revenue did Uber make in 2021?,"{'action': '10K_DOCUMENT_QUERY', 'reason': 'As...",DENIED,NaN,789
4,20:42:45,carol,None,how do I build an agent with the OpenAI Agents...,None,NaN,NaN,0
5,20:42:45,alice,engineer,how do I build an agent with the OpenAI Agents...,"{'action': 'OPENAI_QUERY', 'reason': 'OpenAI A...",Approved,MISS,9169
6,20:42:55,alice,engineer,how do I build an agent with the OpenAI Agents...,"{'action': 'OPENAI_QUERY', 'reason': 'OpenAI A...",Approved,HIT,40165
7,20:46:19,bob,finance_analyst,what was uber revenue in 2021?,"{'action': '10K_DOCUMENT_QUERY', 'reason': 'Re...",Approved,MISS,3254
8,20:46:22,bob,finance_analyst,what was uber revenue in 2021?,"{'action': '10K_DOCUMENT_QUERY', 'reason': 'Re...",Approved,HIT,872
9,20:46:23,alice,engineer,what was uber revenue in 2021?,"{'action': '10K_DOCUMENT_QUERY', 'reason': 'Qu...",DENIED,NaN,801
